# MELTS Parallel Pressure Calculation

This notebook contains code to run the parallelized workflow of the rhyolite-MELTS geobarometer using rhyolite-MELTSv1.0.2. Successful pressures found will be Q1F and/or Q2F. There is also a python script, melts-parallel.py that can be run via terminal commands. Compositions are input via a csv with standard MELTS input format. Example:

| Parameter | KCP-109C | KCP-109A | KCP-109B |
|-----------|----------|----------|----------|
| SiO2 | 75.82 | 76.62 | 76.36 |
| TiO2 | 0.082 | 0.079 | 0.069 |
| Al2O3 | 12.95 | 12.48 | 12.65 |
| Fe2O3 | 0.18 | 0.17 | 0.14 |
| Cr2O3 | | | |
| FeO | 0.49 | 0.47 | 0.39 |
| MnO | 0.031 | 0.026 | 0.024 |
| MgO | 0.09 | 0.06 | 0.05 |
| NiO | | | |
| CoO | | | |
| CaO | 0.7 | 0.53 | 0.54 |
| Na2O | 4.37 | 3.98 | 3.97 |
| K2O | 4.01 | 4.46 | 4.61 |
| P2O5 | 0.009 | 0.004 | 0.006 |
| H2O | 13 | 13 | 13 |
| CO2 | | | |
| SO3 | | | |
| Cl2O-1 | | | |
| F2O-1 | | | |
| | | | |
| Model | rhyolite-MELTS_v1.0.x | rhyolite-MELTS_v1.0.x | rhyolite-MELTS_v1.0.x |
| Calculation | QF_P_Calc | QF_P_Calc | QF_P_Calc |
| T1 | 1100 | 1100 | 1100 |
| T2 | 700 | 700 | 700 |
| ΔT | 1 | 1 | 1 |
| T unit | C | C | C |
| P1 | 400 | 400 | 400 |
| P2 | 50 | 50 | 50 |
| ΔP | 25 | 25 | 25 |
| P unit | MPa | MPa | MPa |
| fO2 offset | 0 | 0 | 0 |
| fO2 buffer | NNO | NNO | NNO |
| fO2 constraint | TRUE | TRUE | TRUE |
| ΔH | 0.5 | 0.5 | 0.5 |
| ΔV | 0 | 0 | 0 |
| ΔS | 0 | 0 | 0 |

Output is in .xlsx files with information from the calculations and the pressure analysis.

The following packages must be installed, dependencies majority used in MeltsHelperFunctions.py:
- thermoengine
- matplotlib==3.10.1
- scipy==1.15.2
- numpy==1.26.4
- futureproof==0.3.1 *used in one implementation in melts-parallel.py, but not necessary for general parallel implementation

## Declare csv filename

In [11]:
CSV_Filename = 'MarvinApliteComps.csv' #Specify your csv filename, standard MELTS input.
import MeltsHelperFunctions as func

In [12]:
import os
print(f"CPU cores: {os.cpu_count()}")

CPU cores: 10


In [13]:
# DETERMINE OPTIMAL WORKER SETTINGS FOR YOUR SYSTEM
# Run this cell to get recommendations for your hardware

# First, check if you have psutil installed

import psutil
import MeltsHelperFunctions as func

# Use the function from MeltsHelperFunctions
max_workers, suggestions = func.calculate_optimal_workers()

print(f"\nFor your system, try one of these configurations:")
print("Example usage:")
for i, (comp, press, total) in enumerate(suggestions[:3], 1):
    print(f"{i}. func.parallel_melts_main_loop(CSV_Filename, max_composition_workers={comp}, max_pressure_workers={press})")
    

    print("psutil not installed. Install with: pip install psutil")
    print("Manual recommendations based on common systems:")
    print("8-core, 16GB: max_composition_workers=2, max_pressure_workers=4")
    print("16-core, 32GB: max_composition_workers=4, max_pressure_workers=4") 
    print("4-core, 8GB: max_composition_workers=1, max_pressure_workers=3")


System: 10 CPU cores, 16.0 GB RAM
Limits: CPU=9, Memory=65
Recommended max total concurrent workers: 9

Recommended configurations:
Composition_workers × Pressure_workers = Total
     1 ×  6 =  6
     2 ×  4 =  8
     3 ×  3 =  9
     4 ×  2 =  8
     5 ×  1 =  5

For your system, try one of these configurations:
Example usage:
1. func.parallel_melts_main_loop(CSV_Filename, max_composition_workers=1, max_pressure_workers=6)
psutil not installed. Install with: pip install psutil
Manual recommendations based on common systems:
8-core, 16GB: max_composition_workers=2, max_pressure_workers=4
16-core, 32GB: max_composition_workers=4, max_pressure_workers=4
4-core, 8GB: max_composition_workers=1, max_pressure_workers=3
2. func.parallel_melts_main_loop(CSV_Filename, max_composition_workers=2, max_pressure_workers=4)
psutil not installed. Install with: pip install psutil
Manual recommendations based on common systems:
8-core, 16GB: max_composition_workers=2, max_pressure_workers=4
16-core, 32G

In [14]:
# COMPLETE PARALLELIZATION WORKFLOW
# Use this instead of the original melts_main_loop functions

def run_parallel_melts_workflow(csv_filename, max_composition_workers=2, max_pressure_workers=4, verbose=False, find_liquidus=True):
    """
    Complete parallelized MELTS workflow that processes multiple compositions
    and handles all file I/O automatically
    """
    if verbose:
        print("="*60)
        print("STARTING PARALLEL MELTS WORKFLOW")
        print("="*60)
    
    # Use the robust parallel function from MeltsHelperFunctions
    results = func.parallel_melts_main_loop(csv_filename, 
                                          max_composition_workers=max_composition_workers,
                                          max_pressure_workers=max_pressure_workers,
                                          verbose=verbose)
    
    # Process and save results
    successful_results = [r for r in results if 'error' not in r]
    failed_results = [r for r in results if 'error' in r]
    
    if verbose:
        print(f"\n{'='*60}")
        print("WORKFLOW SUMMARY:")
        print(f"{'='*60}")
        print(f"✓ Successfully processed: {len(successful_results)} compositions")
        print(f"✗ Failed: {len(failed_results)} compositions")
        
        if successful_results:
            print(f"\nSuccessful files created:")
            for result in successful_results:
                print(f"  - {result['filename']} ({result['num_data_points']} data points)")
        
        if failed_results:
            print(f"\nFailed compositions:")
            for result in failed_results:
                print(f"  - {result['label']}: {result['error']}")
    
    return results


In [15]:
import cProfile

cProfile.run('run_parallel_melts_workflow(CSV_Filename,max_composition_workers=1,max_pressure_workers=8,verbose=False,find_liquidus=True)', 'melts-parallel_1x8.prof') 

/Users/kellylj2/anaconda3/envs/thermoengine310/lib/python3.10/site-packages/thermoengine/equilibrate.py:2132: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  mu_end[i] = np.matmul(c_row, mu_elm) if valid else 0.0
/Users/kellylj2/anaconda3/envs/thermoengine310/lib/python3.10/site-packages/thermoengine/equilibrate.py:2132: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  mu_end[i] = np.matmul(c_row, mu_elm) if valid else 0.0
/Users/kellylj2/anaconda3/envs/thermoengine310/lib/python3.10/site-packages/thermoengine/equilibrate.py:2132: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensu

KeyboardInterrupt: 

2026-01-02 10:35:53,298 - WARNING - wrapper:65 - Timeout occurred in execute after 10 seconds
2026-01-02 10:35:53,299 - WARNING - safe_equilibrium_execute:106 - Equilibrium calculation timed out: T=1023.1K, P=1500.0bar - Function execute timed out after 10 seconds
2026-01-02 10:35:53,299 - ERROR - find_wet_liquidus:446 - Equilibrium calculation failed or timed out at T=750°C
2026-01-02 10:35:53,344 - WARNING - wrapper:65 - Timeout occurred in execute after 10 seconds
2026-01-02 10:35:53,344 - WARNING - safe_equilibrium_execute:106 - Equilibrium calculation timed out: T=1023.1K, P=1750.0bar - Function execute timed out after 10 seconds
2026-01-02 10:35:53,344 - ERROR - find_wet_liquidus:446 - Equilibrium calculation failed or timed out at T=750°C
2026-01-02 10:35:54,623 - WARNING - wrapper:65 - Timeout occurred in execute after 10 seconds
2026-01-02 10:35:54,623 - WARNING - safe_equilibrium_execute:106 - Equilibrium calculation timed out: T=1073.2K, P=500.0bar - Function execute timed o

No check made for phase separation.
No check made for phase separation.


In [ ]:
# READY-TO-RUN PARALLEL EXAMPLE WITH TUNABLE WORKERS
# Uncomment and run this cell to test the parallel workflow
import cProfile

# Example 1: Conservative settings (good for most systems)
cProfile.run('run_parallel_melts_workflow(CSV_Filename,max_composition_workers=3,max_pressure_workers=3,verbose=False,find_liquidus=True)', 'melts-parallel_3x3.prof') 

cProfile.run('run_parallel_melts_workflow(CSV_Filename,max_composition_workers=2,max_pressure_workers=4,verbose=False,find_liquidus=True)', 'melts-parallel_2x4.prof')

cProfile.run('run_parallel_melts_workflow(CSV_Filename,max_composition_workers=1,max_pressure_workers=3,verbose=False,find_liquidus=True)', 'melts-parallel_1x3.prof') 

cProfile.run('run_parallel_melts_workflow(CSV_Filename,max_composition_workers=1,max_pressure_workers=8,verbose=False,find_liquidus=True)', 'melts-parallel_1x8.prof') 
'''
parallel_results = run_parallel_melts_workflow(
    CSV_Filename, 
    max_composition_workers=3,  # Process 2 compositions simultaneously
    max_pressure_workers=3      # Use 4 workers per composition for pressure steps
)
# Total concurrent processes: 2 × 4 = 8

# Example 2: More aggressive (if you have 16+ cores and 32+ GB RAM)
parallel_results = run_parallel_melts_workflow(
    CSV_Filename, 
    max_composition_workers=4,  # Process 4 compositions simultaneously  
    max_pressure_workers=4      # Use 4 workers per composition for pressure steps
)
# Total concurrent processes: 4 × 4 = 16

# Example 3: Single composition, maximum pressure parallelization
parallel_results = run_parallel_melts_workflow(
    CSV_Filename, 
    max_composition_workers=1,  # Process 1 composition at a time
    max_pressure_workers=8      # Use 8 workers for pressure steps
)
# Total concurrent processes: 1 × 8 = 8
'''

print("Uncomment one of the examples above to run the parallel workflow.")
print("Use the cell above this one to determine optimal worker settings for your system.")
